## Install packages



In [18]:
!pip install -q langchain langchain-community chromadb sentence-transformers groq

## Set up API

In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)

print(response.choices[0].message.content)

## Load Documents

In [22]:
import urllib.request

base_url = "https://raw.githubusercontent.com/Dazcam/applied-ai-portfolio/main/01-rag-pipeline/data/"

files = {
    "compliance": "highland_brew_compliance.txt",
    "suppliers": "highland_brew_suppliers.txt",
    "staff": "highland_brew_staff.txt"
}

documents = {}
for key, filename in files.items():
    url = base_url + filename
    with urllib.request.urlopen(url) as response:
        documents[key] = response.read().decode("utf-8")
    print(f"Loaded {filename} — {len(documents[key])} characters")

Loaded highland_brew_compliance.txt — 3852 characters
Loaded highland_brew_suppliers.txt — 2042 characters
Loaded highland_brew_staff.txt — 2157 characters


## Step 1 splitting

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialise the splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # each chunk is max 500 characters
    chunk_overlap=50,      # chunks overlap by 50 characters to preserve context
    separators=["\n\n", "\n", ".", " "]  # split on paragraphs first, then lines, then sentences
)

# Split each document and tag each chunk with its source
all_chunks = []
for source, text in documents.items():
    chunks = splitter.create_documents(
        texts=[text],
        metadatas=[{"source": source}]  # tag so we know which doc each chunk came from
    )
    all_chunks.extend(chunks)
    print(f"{source}: {len(chunks)} chunks")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

compliance: 9 chunks
suppliers: 6 chunks
staff: 6 chunks

Total chunks across all documents: 21


## Step 2 - Embedding

In [25]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Test it on a single sentence so we can see what an embedding looks like
test_embedding = embeddings.embed_query("What is the shelf life of the cider?")

print(f"Embedding length: {len(test_embedding)} dimensions")
print(f"First 5 values: {test_embedding[:5]}")

/tmp/ipykernel_12327/1218697798.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding length: 384 dimensions
First 5 values: [-0.07951623946428299, -0.010083423927426338, -0.12163972854614258, 0.054590534418821335, 0.04542642831802368]


## Step 3 - Storing in ChromaDB

In [26]:
from langchain_community.vectorstores import Chroma

# Create the vector store from our chunks
# This embeds all 21 chunks and stores them in ChromaDB
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="highland_brew"
)

print(f"Vector store created successfully")
print(f"Total vectors stored: {vectorstore._collection.count()}")

Vector store created successfully
Total vectors stored: 21
